# RT-DETR-L (Ultralytics)

Download → shared preprocessing/augmentation → training → evaluation for `rtdetr-l.pt`. RT-DETR has no NMS; the evaluation keeps queries scoring ≥ 0.001, at most 100 per image.

In [ ]:
import os

from ultralytics import RTDETR

from ppe_detection_evaluation import RTDETR_L
from ppe_detection_evaluation import ultralytics_models as um
from ppe_detection_evaluation.augment import IMAGE_SIZE, SharedAugmentation, build_train_transform
from ppe_detection_evaluation.data import (
    check_dataset,
    class_box_stats,
    download_dataset,
    verify_same_examples,
    write_ultralytics_yaml,
)
from ppe_detection_evaluation.experiment import BATCH_SIZE, DATA_ROOT, EPOCHS, RESULTS_DIR, SEED
from ppe_detection_evaluation.visualize import plot_examples

## 1. Download the dataset

Construction Site Safety **v30** from Roboflow, exported in YOLO TXT format. This release has no augmentations baked
in (unlike the Kaggle mirror, which already contains mosaics), so the shared pipeline below is the only augmentation
any model sees.

Set `ROBOFLOW_API_KEY` in the environment before the first run. `download_dataset` is a no-op once
`data/construction-site-safety-v30/data.yaml` exists, so all four notebooks share one copy.

In [ ]:
download_dataset(DATA_ROOT, api_key=os.environ.get("ROBOFLOW_API_KEY", ""))
summary = check_dataset(DATA_ROOT)  # raises if the class order differs from DEFAULT_CLASS_NAMES
summary

`roboflow_notes` should list only auto-orient / resize to 640×640 and **no augmentation**. The box-size table below is
the gate from AUGMENTATIONS.md §6 Step 0: if more than ~15 % of `Mask` / `NO-Mask` boxes are under 16 px, raise the
minimum of `SCALE_RANGE` from 0.75 to 0.85 in `augment.py` (for every model, not just this one).

In [ ]:
stats = class_box_stats(DATA_ROOT, "train")
for name, s in stats.items():
    print(f"{name:15s} n={s['count']:5d}  <16px {s['under_16px']:6.1%}  <32px {s['under_32px']:6.1%}")

In [ ]:
# Roboflow's data.yaml has relative paths that Ultralytics resolves against the CWD; write an absolute one.
data_yaml = write_ultralytics_yaml(DATA_ROOT)
print(data_yaml.read_text())

## 2. Preprocessing and augmentation

All four models import the same pipeline from `ppe_detection_evaluation.augment`:

| # | Augmentation | Parameters | p |
|---|---|---|---|
| A1 | Horizontal flip | — | 0.5 |
| A2 | Scale + translate | scale 0.75–1.25, translate ±10 %, gray fill 114 | 1.0 |
| A3 | Colour jitter | brightness ±0.4, contrast ±0.3, saturation ±0.5, hue ±0.015 | 1.0 |
| A4 | One of blur / noise / JPEG | σ 0.1–1.5; σ 0.02–0.06; quality 40–90 | 0.25 |

Boxes keeping less than 40 % of their area, or narrower than 3 px, are dropped. Validation and test images only get
the deterministic preprocessing (load → RGB → 640×640, a no-op for this export).

**Same examples for every model.** `SharedAugmentation` reseeds the pipeline from `(SEED, epoch, image name)` before
each image, so the augmented example is independent of the framework, DataLoader workers, batch size and shuffle
order. With the shared `SEED` and `EPOCHS` from `experiment.py`, every model trains on exactly the same sequence of
examples; only the order within an epoch differs.

In [ ]:
build_train_transform()

Ultralytics' own augmentation stack (mosaic, HSV, random perspective, flips, its default Albumentations blur /
grayscale / CLAHE) is bypassed: `um.SharedAugmentationRTDETRTrainer` trains on a dataset whose `__getitem__` calls the same
`load_training_example` as the torchvision models, then runs only Ultralytics' eval-mode letterbox (a no-op at 640×640)
and `Format`. The built-in augmentation arguments are still zeroed (`um.DISABLE_BUILTIN_AUGMENTATION`) so the run's
`args.yaml` records that, and because `multi_scale` would otherwise rescale whole batches.

A callback restarts the persistent DataLoader workers at the start of each epoch so they pick up the new epoch number.

In [ ]:
um.DISABLE_BUILTIN_AUGMENTATION

In [ ]:
train_ds = um.build_train_dataset(data_yaml, seed=SEED, rtdetr=True)
len(train_ds)

Four training images (columns) as the model sees them in epochs 0–2 (rows), taken from the Ultralytics dataset itself:

In [ ]:
examples, titles = [], []
for epoch in range(3):
    train_ds.epoch = epoch
    for i in range(4):
        path, image, boxes, labels = um.as_raw_example(train_ds, i)
        examples.append((image, boxes, labels))
        titles.append(f"epoch {epoch} · {os.path.basename(path)[:20]}")
train_ds.epoch = 0
plot_examples(examples, titles);

### Check: this model is fed the shared examples

`verify_same_examples` pulls samples through **this model's own training dataset**, converts them back to RGB uint8
images with pixel boxes, and asserts they equal the output of the shared pipeline. The fingerprints printed here must
be identical in all four model notebooks.

In [ ]:
augmentation = SharedAugmentation(seed=SEED)
fingerprints = {}
for epoch in (0, 1):
    train_ds.epoch = epoch
    fingerprints[epoch] = verify_same_examples(
        lambda i: um.as_raw_example(train_ds, i), range(min(32, len(train_ds))), augmentation, epoch
    )
train_ds.epoch = 0
fingerprints

## 3. Model

COCO-pretrained `rtdetr-l.pt`; Ultralytics replaces the head for the 10 classes in `data_yaml`.

In [ ]:
model = RTDETR("rtdetr-l.pt")
model.info()

## 4. Training

Not run in this notebook. `seed=SEED` is also the shared augmentation seed (the dataset reads `args.seed`).
Optimiser, learning-rate schedule and loss stay at Ultralytics' defaults for this model.

After training starts, open `train_batch0.jpg` in the run directory: it must show single images with mild scale,
colour and blur/noise changes, and **no** mosaic collages or grayscale frames.

In [ ]:
model.train(
    trainer=um.SharedAugmentationRTDETRTrainer,
    data=str(data_yaml),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMAGE_SIZE,
    seed=SEED,
    project=str(RESULTS_DIR / "runs"),
    name="rtdetr_l",
    exist_ok=True,
    **um.DISABLE_BUILTIN_AUGMENTATION,
)

## 5. Evaluation

The best checkpoint (by Ultralytics' validation fitness) is run on the eval-transformed test images and scored with
the `EvaluationFramework`, exactly like the torchvision models. `model.predict` gets BGR arrays at 640×640 with
`conf=0.001` and `max_det=100`; latency is the sum of Ultralytics' pre-process, inference and post-process times.

In [ ]:
best = RTDETR(RESULTS_DIR / "runs" / "rtdetr_l" / "weights" / "best.pt")
fw = um.evaluate(best, DATA_ROOT, "test", RTDETR_L)
results = fw.evaluate()
fw.save(RESULTS_DIR / "rtdetr_l_test.json")

In [ ]:
det = results["detection"]["groups"]
print(f"mAP@[.5:.95] {det['all']['mAP']:.4f}   mAP@.5 {det['all']['mAP50']:.4f}")
print(f"PPE mAP@.5 {det['ppe']['mAP50']:.4f}   small PPE mAP@.5 {results['detection']['by_size']['small']['groups']['ppe']['mAP50']:.4f}")
print("violation mean:", results["violation"]["mean"])
print("latency:", results["latency"])